In [1]:
# Project Paths + Basic Imports

import os
import numpy as np
import tensorflow as tf

BASE_DIR = r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-3_REST_Image_Service"
TRAIN_DIR = os.path.join(BASE_DIR, "dataset", "train")
VAL_DIR   = os.path.join(BASE_DIR, "dataset", "validation")
MODEL_DIR = os.path.join(BASE_DIR, "model")

os.makedirs(MODEL_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Train dir:", TRAIN_DIR)
print("Val dir:", VAL_DIR)
print("Model dir:", MODEL_DIR)
print("GPU:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.20.0
Train dir: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-3_REST_Image_Service\dataset\train
Val dir: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-3_REST_Image_Service\dataset\validation
Model dir: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-3_REST_Image_Service\model
GPU: []


In [2]:
# Project Configuration + Image Generators

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("Class indices:", train_generator.class_indices)


Found 192 images belonging to 3 classes.
Found 48 images belonging to 3 classes.
Class indices: {'bike': 0, 'bus': 1, 'car': 2}


In [3]:
# Build Transfer Learning Model (MobileNetV2)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # Freeze base model

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
# Train Frozen Model

EPOCHS = 8

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)


c:\Users\hasan\AppData\Local\Programs\Python\Python310\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/8
12/12 ━━━━━━━━━━━━━━━━━━━━ 37s 3s/step - accuracy: 0.6406 - loss: 0.8651 - val_accuracy: 0.8125 - val_loss: 0.4111
Epoch 2/8
12/12 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.8802 - loss: 0.3170 - val_accuracy: 0.8333 - val_loss: 0.4068
Epoch 3/8
12/12 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.9271 - loss: 0.2259 - val_accuracy: 0.8958 - val_loss: 0.2703
Epoch 4/8
12/12 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step - accuracy: 0.9531 - loss: 0.1602 - val_accuracy: 0.8958 - val_loss: 0.2886
Epoch 5/8
12/12 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.9688 - loss: 0.0957 - val_accuracy: 0.8958 - val_loss: 0.2616
Epoch 6/8
12/12 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.9635 - loss: 0.0958 - val_accuracy: 0.8958 - val_loss: 0.2601
Epoch 7/8
12/12 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.9688 - loss: 0.0877 - val_accuracy: 0.9167 - val_loss: 0.2578
Epoch 8/8
12/12 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.9635 - loss: 0.0872 - val_accuracy: 0.8750 - val_loss: 0.2642


In [5]:
# Save Trained Model

model_path = os.path.join(MODEL_DIR, "vehicle_classifier_mobilenetv2.h5")
model.save(model_path)

print("Model saved at:", model_path)


Model saved at: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-3_REST_Image_Service\model\vehicle_classifier_mobilenetv2.h5


In [6]:
# Fine-Tuning Setup (Unfreeze last layers + Lower LR)

# Unfreeze the last ~30 layers of the base model
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Re-compile with a lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Base model trainable:", base_model.trainable)
print("Trainable layers:", sum([1 for l in model.layers if l.trainable]))


Base model trainable: True
Trainable layers: 5


In [7]:
# Fine-Tuning Training

FINE_TUNE_EPOCHS = 5

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=FINE_TUNE_EPOCHS
)


Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 46s 3s/step - accuracy: 0.8385 - loss: 0.3572 - val_accuracy: 0.8750 - val_loss: 0.2628
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.8906 - loss: 0.2716 - val_accuracy: 0.8750 - val_loss: 0.2617
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.9323 - loss: 0.2040 - val_accuracy: 0.8958 - val_loss: 0.2607
Epoch 4/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - accuracy: 0.9635 - loss: 0.1294 - val_accuracy: 0.8958 - val_loss: 0.2613
Epoch 5/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.9219 - loss: 0.2037 - val_accuracy: 0.8958 - val_loss: 0.2625


In [8]:
# Save Fine-Tuned Model

finetuned_path = os.path.join(MODEL_DIR, "vehicle_classifier_mobilenetv2_finetuned.h5")
model.save(finetuned_path)

print("Fine-tuned model saved at:", finetuned_path)


Fine-tuned model saved at: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 4\Task4-3_REST_Image_Service\model\vehicle_classifier_mobilenetv2_finetuned.h5
